## Settings

In [1]:
## auto reload modules
%reload_ext autoreload
%autoreload 2

## Dependencies

In [2]:
## libraries
import sys
import pickle
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.data.builders import load_processed_data
from src.estimators.factories import load_estimators
from src.evaluators.caching import load_notebook_cache
from src.evaluators.metrics import spec_marginal_delta
from src.evaluators.tables import main_table
from src.evaluators.decomposing import (
    train_decomposed_separation,
    train_decomposed_attribution,
    train_decomposed_consensus,
    compile_decomposed_separation,
    compile_decomposed_attribution,
    compile_decomposed_consensus,
    compile_decomposed_full,
    stat_decomposed_summary,
    stat_decomposed_attribution,
    stat_decomposed_test,
    )

## constants
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z,
    TARGET
)

## Initialization

In [3]:
## reproducibility
N_DECIMALS = 2
RANDOM_STATE = 42
N_REPEATS = 30
FORCE_RECOMPUTE = False

## load data and models
data = load_processed_data()
models = load_estimators(random_state = RANDOM_STATE)

## view data shape
n_obs, n_feat = data.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")

## view model surface
n_mods = len(models)
print(f"Learned Models: {n_mods} estimators")

## cache setup
cache_path = root / "notebooks" / "cache" / "ablate_results.pkl"
cache_metadata = {
    "n_obs": len(data),
    "n_repeats": N_REPEATS,
    "random_state": RANDOM_STATE,
    "model_names": sorted(models.keys()),
    "target": TARGET,
    "feat_x": list(FEAT_X),
    "feat_z": list(FEAT_Z),
    "attribution_protocol": "nested_logo_v1",
    "specification_schema": ("invariants", "signatures"),
}
cache_keys = (
    "results_decomposed_attribution",
    "predictions_decomposed_attribution",
    "results_decomposed_separation",
    "predictions_decomposed_separation",
    "results_decomposed_consensus",
)
cache_payload = load_notebook_cache(
    cache_path = cache_path,
    metadata = cache_metadata,
    required_keys = cache_keys,
    force_recompute = FORCE_RECOMPUTE,
)
if cache_payload is not None:
    globals().update({key: cache_payload[key] for key in cache_keys})

Original Data: 32 features, 25 observations
Learned Models: 9 estimators
Forced recomputation for /Users/nickkunz/Documents/Nick’s MacBook Pro/Python/network-capacity/notebooks/cache/ablate_results.pkl


## Training

In [4]:
## train decomposed residual attribution
if cache_payload is None:
    results_dict_decomposed_attribution = train_decomposed_attribution(
        data = data,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE
    )

Residual attribution evaluation:   0%|          | 0/9 [00:00<?, ?model/s]

Residual attribution evaluation:  11%|█         | 1/9 [00:06<00:54,  6.87s/model]

Residual attribution evaluation:  33%|███▎      | 3/9 [00:09<00:16,  2.69s/model]

Residual attribution evaluation:  44%|████▍     | 4/9 [00:24<00:35,  7.10s/model]

Residual attribution evaluation:  56%|█████▌    | 5/9 [00:32<00:29,  7.43s/model]

Residual attribution evaluation:  67%|██████▋   | 6/9 [00:47<00:28,  9.65s/model]

Residual attribution evaluation:  78%|███████▊  | 7/9 [04:45<02:42, 81.47s/model]

Residual attribution evaluation:  89%|████████▉ | 8/9 [04:50<00:57, 57.92s/model]

Residual attribution evaluation: 100%|██████████| 9/9 [04:52<00:00, 40.61s/model]

Residual attribution evaluation: 100%|██████████| 9/9 [04:52<00:00, 32.48s/model]

In [5]:
## train decomposed separation
if cache_payload is None:
    results_dict_decomposed_separation = train_decomposed_separation(
        data = data,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE
    )

Separation evaluation:   0%|          | 0/9 [00:00<?, ?model/s]

Separation evaluation:  11%|█         | 1/9 [00:05<00:41,  5.17s/model]

Separation evaluation:  22%|██▏       | 2/9 [00:07<00:24,  3.56s/model]

Separation evaluation:  33%|███▎      | 3/9 [00:09<00:16,  2.71s/model]

Separation evaluation:  44%|████▍     | 4/9 [00:37<01:03, 12.69s/model]

...

Separation evaluation:  67%|██████▋   | 6/9 [01:05<00:39, 13.16s/model]

Separation evaluation:  78%|███████▊  | 7/9 [04:40<02:38, 79.10s/model]

Separation evaluation:  89%|████████▉ | 8/9 [04:56<00:59, 59.18s/model]

Separation evaluation: 100%|██████████| 9/9 [05:03<00:00, 42.89s/model]

Separation evaluation: 100%|██████████| 9/9 [05:03<00:00, 33.74s/model]

## Post-Processing

In [6]:
if cache_payload is None:
    results_decomposed_attribution, predictions_decomposed_attribution = compile_decomposed_attribution(
        results = results_dict_decomposed_attribution,
    )
    results_decomposed_separation, predictions_decomposed_separation = compile_decomposed_separation(
        results = results_dict_decomposed_separation,
    )

if cache_payload is None or "results_decomposed_full_agreement" not in cache_payload:
    predictions_decomposed_consensus = train_decomposed_consensus(
        data = data,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE,
        n_jobs = -1,
    )
    results_decomposed_consensus = compile_decomposed_consensus(
        predictions = predictions_decomposed_consensus,
    )
    results_decomposed_full_agreement = compile_decomposed_full(
        predictions = predictions_decomposed_consensus,
    )
    cache_path.parent.mkdir(parents = True, exist_ok = True)
    cache_payload = {
        **(cache_payload or {}),
        "metadata": cache_metadata,
        "results_decomposed_attribution": results_decomposed_attribution,
        "predictions_decomposed_attribution": predictions_decomposed_attribution,
        "results_decomposed_separation": results_decomposed_separation,
        "predictions_decomposed_separation": predictions_decomposed_separation,
        "predictions_decomposed_consensus": predictions_decomposed_consensus,
        "results_decomposed_consensus": results_decomposed_consensus,
        "results_decomposed_full_agreement": results_decomposed_full_agreement,
    }
    cache_path.write_bytes(pickle.dumps(cache_payload))
    print(f"Cached ablation results -> {cache_path}")

Cached ablation results -> /Users/nickkunz/Documents/Nick’s MacBook Pro/Python/network-capacity/notebooks/cache/ablate_results.pkl


## Residual Attribution Test
The residual attribution test probes whether, after estimating the capacity stage $C(X)$, the remaining slack is better predicted from dynamics than from topology. Under the additive decomposition $y^* = C(X) + R(Z)$, the residual slack $s = y^* - \hat{C}(X)$ should be more predictable from $Z$ than from $X$. Each contrast is evaluated with a paired one-sided Wilcoxon signed-rank statistic on domain-level mean absolute error (MAE).

- **$H_0$**: Topology predicts residual slack at least as well as dynamics ($\Delta \mathrm{MAE} \le 0$).
- **$H_1$**: Dynamics predict residual slack better than topology ($\Delta \mathrm{MAE} > 0$).

Rejecting $H_0$ establishes that residual slack is more predictably attributable to dynamics than to unrecovered topological information after conditioning on $C(X)$.

### Decomposition Protocol
Separate learners are fit for $X \to s$ and $Z \to s$ within repeated-seed LOGO-CV (domain) splits. Predictions are averaged across seeds, then aggregated to one MAE value per (model $\times$ domain) for each residual-input source.

### Empirical Test Specification
Paired differences are formed as $\Delta \mathrm{MAE} = \mathrm{MAE}_{X \to s} - \mathrm{MAE}_{Z \to s}$ on matched (model $\times$ domain) units. The table reports the paired median shift, rank-biserial effect size, and Holm-adjusted one-sided p-value summarizing whether the dynamics-based residual model consistently outperforms the topology-based residual model.

In [7]:
## one-sided wilcoxon test for residual attribution
results_attribution = stat_decomposed_attribution(
    predictions = predictions_decomposed_attribution,
    decimals = N_DECIMALS,
)

display(results_attribution)

Paired One-Sided Test (Wilcoxon Signed-Rank): n = 45
H₀: Δ MAE ≤ 0
H₁: Δ MAE > 0
Median Δ MAE: Median of paired differences, not the difference of marginal medians
Rank-biserial r: Raw paired effect size; positive values indicate lower error for Z -> slack
One-sided p: Wilcoxon signed-rank p-value for H₁
Holm-adj. p: Holm-Bonferroni adjusted one-sided p-value
Diff.: Yes if Holm-adj. p < 0.05 and Median Δ MAE > 0
Significance codes reflect Holm-adj. p
*** p < 0.001, ** p < 0.01, * p < 0.05


,,Median Δ MAE,Rank-biserial r,One-sided p,Holm-adj. p,Sig.,Diff.
Property,Comparison,,,,,,
Residual Attribution,Structure vs Dynamics,0.52,0.33,0.026,0.026,*,Yes


_Residual slack is predicted more accurately from dynamics than from topology, with a positive median MAE shift and a significant one-sided Wilcoxon result. This means the additive decomposition is not splitting predictors into arbitrary blocks. After the capacity component $C(X)$ is removed, the remaining explainable variation aligns with dynamic information rather than leftover topological signal. The result supports a clear division: topology carries baseline capacity, and dynamics carry the residual structure that remains after capacity is accounted for._

## Parsimonious Sufficiency Tests
The parsimonious sufficiency tests probe whether the log-additive decomposition $C(X) + R(Z)$ is non-inferior to all considered test specifications, both richer and simpler. Two performance dimensions are evaluated independently for each comparison: the efficiency index (EI) measures frontier placement accuracy, and the consensus index (CI) measures rank-order agreement.

### Unified Non-Inferiority Claim
For every test specification -- interaction-augmented residual, joint, capacity-only, and dynamics-only -- the claim is that the test specification does not meaningfully outperform the original log-additive decomposition. Each test specification is compared against the original log-additive decomposition with a paired one-sided Wilcoxon signed-rank statistic under an empirically derived margin $\delta$.

- **$H_0$**: The test specification exceeds the original log-additive decomposition by at least $\delta$ ($\Delta \ge \delta$).
- **$H_1$**: The test specification does not exceed the original log-additive decomposition by $\delta$ ($\Delta < \delta$).

Rejecting $H_0$ establishes that the test specification fails to produce a practically meaningful improvement over the original log-additive form. Simpler specifications, which are expected to perform no better than the log-additive decomposition, satisfy the same test trivially when their paired gaps lie at or below the original log-additive level.

### Decomposition Protocol
All specifications are fit under repeated-seed LOGO-CV (domain): original log-additive decomposition $C(X) + R(Z)$, interaction-augmented residual, joint, capacity-only, and dynamics-only. Predictions are averaged across seeds before scoring per (model $\times$ domain).

### Reporting Convention
The descriptive summary reports median `[IQR]` EI across matched (model $\times$ domain) units, with `VR`, `MV`, and `MS` included as auxiliary frontier diagnostics. This table provides the empirical scale for the tests below.

### Empirical Test Specification
The margin $\delta$ is set from the interquartile range of original log-additive decomposition values, anchoring the test to natural variability under the log-additive decomposition (full IQR for both EI and CI, each floored to the second decimal). Paired gaps are formed as $\Delta = \text{score}_{\text{test}} - \text{score}_{\text{original}}$, and each table reports the median paired gap, rank-biserial effect size, Holm-adjusted one-sided p-value, and decision for each test specification.

In [8]:
## empirical delta for EI tests
delta_ei = spec_marginal_delta(
    results = results_decomposed_separation,
    feat_value = ["ei"],
    label_ref = "specification",
    value_ref = "additive",
    method = "iqr",
    scale = 1.0,  ## full iqr range
    decimals = N_DECIMALS
)

## empirical delta for CI tests
delta_ci = spec_marginal_delta(
    results = results_decomposed_separation,
    feat_value = ["ci"],
    label_ref = "specification",
    value_ref = "additive",
    method = "iqr",
    scale = 1.00,  ## full iqr range
    decimals = N_DECIMALS
)


In [9]:
## non-inferiority test: richer specifications on EI
results_complex_ei = stat_decomposed_test(
    results = results_decomposed_separation,
    delta = delta_ei,
    metric = "ei",
    specs = ("joint", "interaction"),
    direction = "noninferiority",
    decimals = N_DECIMALS)

display(results_complex_ei)

Paired Non-Inferiority Test (Wilcoxon Signed-Rank): n = 45, δ = 0.08, metric = EI
H₀: Δ EI ≥ δ
H₁: Δ EI < δ
NI.: Yes if Holm-adj. p < 0.05 and Median Δ EI < δ
Median Δ EI: Median of paired differences (Test - Original)
Rank-biserial r: Raw paired effect size, positive values indicate specification > additive
One-sided p: Wilcoxon signed-rank p-value for H₁
Holm-adj. p: Holm-Bonferroni adjusted one-sided p-value
Significance codes reflect Holm-adj. p
*** p < 0.001, ** p < 0.01, * p < 0.05


,Median Δ EI,Rank-biserial r,One-sided p,Holm-adj. p,Sig.,NI.
Ablation,,,,,,
Joint,-0.02,-0.55,<0.001,<0.001,***,Yes
Interaction,-0.01,-0.46,<0.001,<0.001,***,Yes


In [10]:
## non-inferiority test: complex specifications on CI
results_complex_ci = stat_decomposed_test(
    results = results_decomposed_separation,
    delta = delta_ci,
    metric = "ci",
    specs = ("joint", "interaction"),
    direction = "noninferiority",
    decimals = N_DECIMALS
)
display(results_complex_ci)

Paired Non-Inferiority Test (Wilcoxon Signed-Rank): n = 45, δ = 0.14, metric = CI
H₀: Δ CI ≥ δ
H₁: Δ CI < δ
NI.: Yes if Holm-adj. p < 0.05 and Median Δ CI < δ
Median Δ CI: Median of paired differences (Test - Original)
Rank-biserial r: Raw paired effect size, positive values indicate specification > additive
One-sided p: Wilcoxon signed-rank p-value for H₁
Holm-adj. p: Holm-Bonferroni adjusted one-sided p-value
Significance codes reflect Holm-adj. p
*** p < 0.001, ** p < 0.01, * p < 0.05


,Median Δ CI,Rank-biserial r,One-sided p,Holm-adj. p,Sig.,NI.
Ablation,,,,,,
Joint,-0.27,-0.67,<0.001,<0.001,***,Yes
Interaction,-0.10,-0.65,<0.001,<0.001,***,Yes


In [11]:
## non-inferiority test: simple specifications on EI
results_simple_ei = stat_decomposed_test(
    results = results_decomposed_separation,
    delta = delta_ei,
    metric = "ei",
    specs = ("invariants", "signatures"),
    direction = "noninferiority",
    decimals = N_DECIMALS
)

display(results_simple_ei)

Paired Non-Inferiority Test (Wilcoxon Signed-Rank): n = 45, δ = 0.08, metric = EI
H₀: Δ EI ≥ δ
H₁: Δ EI < δ
NI.: Yes if Holm-adj. p < 0.05 and Median Δ EI < δ
Median Δ EI: Median of paired differences (Test - Original)
Rank-biserial r: Raw paired effect size, positive values indicate specification > additive
One-sided p: Wilcoxon signed-rank p-value for H₁
Holm-adj. p: Holm-Bonferroni adjusted one-sided p-value
Significance codes reflect Holm-adj. p
*** p < 0.001, ** p < 0.01, * p < 0.05


,Median Δ EI,Rank-biserial r,One-sided p,Holm-adj. p,Sig.,NI.
Ablation,,,,,,
Invariants,-0.01,-0.77,<0.001,<0.001,***,Yes
Signatures,-0.01,-0.21,<0.001,<0.001,***,Yes


In [12]:
## non-inferiority test: simple specifications on CI
results_simple_ci = stat_decomposed_test(
    results = results_decomposed_separation,
    delta = delta_ci,
    metric = "ci",
    specs = ("invariants", "signatures"),
    direction = "noninferiority",
    decimals = N_DECIMALS
)
display(results_simple_ci)

Paired Non-Inferiority Test (Wilcoxon Signed-Rank): n = 45, δ = 0.14, metric = CI
H₀: Δ CI ≥ δ
H₁: Δ CI < δ
NI.: Yes if Holm-adj. p < 0.05 and Median Δ CI < δ
Median Δ CI: Median of paired differences (Test - Original)
Rank-biserial r: Raw paired effect size, positive values indicate specification > additive
One-sided p: Wilcoxon signed-rank p-value for H₁
Holm-adj. p: Holm-Bonferroni adjusted one-sided p-value
Significance codes reflect Holm-adj. p
*** p < 0.001, ** p < 0.01, * p < 0.05


,Median Δ CI,Rank-biserial r,One-sided p,Holm-adj. p,Sig.,NI.
Ablation,,,,,,
Invariants,-0.20,-0.86,<0.001,<0.001,***,Yes
Signatures,-0.08,-0.55,<0.001,<0.001,***,Yes


In [13]:
## median decomposition metrics by specification
DECOMPOSITION_REPORT_SPECS = (
    "additive",
    "invariants",
    "signatures",
    "joint",
    "interaction",
)

In [14]:
## decomposed results report by specification
results_decomposed_report = results_decomposed_separation.loc[
    results_decomposed_separation["specification"].isin(DECOMPOSITION_REPORT_SPECS)
].copy()

decomposed_summary = stat_decomposed_summary(
    results = results_decomposed_report,
    metric = "ei",
    spec_order = DECOMPOSITION_REPORT_SPECS,
    decimals = N_DECIMALS,
)

display(decomposed_summary)


EI [IQR]   VR   MV    MS
Ablation Method                                        
Original Original     0.76 [0.71, 0.80]  0.0  0.0  6.07
Simple   Invariants   0.74 [0.70, 0.77]  0.0  0.0  7.10
         Signatures   0.76 [0.71, 0.79]  0.0  0.0  5.31
Complex  Joint        0.75 [0.70, 0.77]  0.0  0.0  6.38
         Interaction  0.74 [0.69, 0.79]  0.0  0.0  5.92

In [15]:
## median decomposition metrics by specification (CI)
results_decomposed_report = results_decomposed_separation.loc[
    results_decomposed_separation["specification"].isin(DECOMPOSITION_REPORT_SPECS)
].copy()

decomposed_summary_ci = stat_decomposed_summary(
    results = results_decomposed_report,
    metric = "ci",
    spec_order = DECOMPOSITION_REPORT_SPECS,
    decimals = N_DECIMALS,
)

display(decomposed_summary_ci)


CI [IQR]  RHO   RBO   DCR
Ablation Method                                         
Original Original     0.84 [0.77, 0.92]  0.6  0.86  0.78
Simple   Invariants   0.57 [0.00, 0.72]  0.0  0.84  0.52
         Signatures   0.59 [0.00, 0.87]  0.0  0.87  0.51
Complex  Joint        0.58 [0.00, 0.80]  0.0  0.86  0.54
         Interaction  0.71 [0.60, 0.80]  0.5  0.84  0.64

_No alternative specification — richer or simpler — improves on the additive form by a practically meaningful margin on either EI or CI. Each Holm-adjusted one-sided Wilcoxon test rejects the hypothesis that the alternative outperforms additive by at least $\delta$, with simpler specifications (capacity-only and dynamics-only) sitting at or below the additive level and richer specifications (interaction, joint) showing no consistent gain. Together, these tests support $C(X) + R(Z)$ as a sufficient specification: added complexity does not pay off, and reduced specifications do not surpass it. Topology carries baseline capacity, dynamics carry the remaining structured variation, and the additive form stands as the parsimonious decomposition._

## Summary

In [16]:
## manuscript summary table for decomposition validity
def _prep_ablation_table(results: pd.DataFrame, category: str, decision_col: str = "NI.") -> pd.DataFrame:
    frame = results.reset_index().copy()
    frame = frame.rename(columns = {"Ablation": "Method", decision_col: "Decision"})
    frame["Ablation"] = category
    return frame

## ei side (domain transfer)
summary_ei = pd.concat(
    objs = [
        _prep_ablation_table(results = results_complex_ei, category = "Complex"),
        _prep_ablation_table(results = results_simple_ei, category = "Simple"),
    ],
    ignore_index = True,
)

## ci side (prediction consensus)
summary_ci = pd.concat(
    objs = [
        _prep_ablation_table(results = results_complex_ci, category = "Complex"),
        _prep_ablation_table(results = results_simple_ci, category = "Simple"),
    ],
    ignore_index = True,
)

display_table = main_table(
    results_left = summary_ei,
    results_right = summary_ci,
    index_feat = ["Ablation", "Method"],
    value_feat_left = ["Median Δ EI", "Rank-biserial r", "Sig.", "Decision"],
    value_feat_right = ["Median Δ CI", "Rank-biserial r", "Sig.", "Decision"],
    header_left = "Domain Transfer",
    header_right = "Estimation-Observation Consensus",
    short_feat_left = ["Δ EI", "r", "Sig.", "N/I"],
    short_feat_right = ["Δ CI", "r", "Sig.", "N/I"],
    row_order = [
        ("Simple", "Invariants"),
        ("Simple", "Signatures"),
        ("Complex", "Joint"),
        ("Complex", "Interaction")
    ],
)

display(display_table)
display(
    Markdown(
        "**Δ EI**, **Δ CI**: median paired difference Δ = Test - Original, where Test is the listed ablation and Original is the log-additive decomposition; negative values favor the original log-additive decomposition. "
        "**r**: rank-biserial correlation effect size, with positive values favoring the tested one-sided claim. "
        "**Sig.**: Holm-adjusted one-sided Wilcoxon signed-rank *p* below threshold within each outcome "
        "(\*\*\* *p* < 0.001, \*\* *p* < 0.01, \* *p* < 0.05); significance supports the unified non-inferiority claim H₁: Δ < δ, that the test ablation does not outperform the original log-additive decomposition by the margin. "
        "**Non-Inferior**: additive is supported as non-inferior when Holm-adjusted *p* < 0.05 and median Δ < δ. "
        "Margin δ = IQR derived from natural variability in the log-additive decomposition (full IQR for both EI and CI). "
        "Each test is conducted on *n* = 45 (9 models × 5 domains)."
    )
)

Domain Transfer                   \
                                Δ EI      r Sig.  N/I   
Ablation Method                                         
Simple   Invariants            -0.01  -0.77  ***  Yes   
         Signatures            -0.01  -0.21  ***  Yes   
Complex  Joint                 -0.02  -0.55  ***  Yes   
         Interaction           -0.01  -0.46  ***  Yes   

                     Estimation-Observation Consensus                   
                                                 Δ CI      r Sig.  N/I  
Ablation Method                                                         
Simple   Invariants                             -0.20  -0.86  ***  Yes  
         Signatures                             -0.08  -0.55  ***  Yes  
Complex  Joint                                  -0.27  -0.67  ***  Yes  
         Interaction                            -0.10  -0.65  ***  Yes

**Δ EI**, **Δ CI**: median paired difference Δ = Test - Original, where Test is the listed ablation and Original is the log-additive decomposition; negative values favor the original log-additive decomposition. **r**: rank-biserial correlation effect size, with positive values favoring the tested one-sided claim. **Sig.**: Holm-adjusted one-sided Wilcoxon signed-rank *p* below threshold within each outcome (\*\*\* *p* < 0.001, \*\* *p* < 0.01, \* *p* < 0.05); significance supports the unified non-inferiority claim H₁: Δ < δ, that the test ablation does not outperform the original log-additive decomposition by the margin. **Non-Inferior**: additive is supported as non-inferior when Holm-adjusted *p* < 0.05 and median Δ < δ. Margin δ = IQR derived from natural variability in the log-additive decomposition (full IQR for both EI and CI). Each test is conducted on *n* = 45 (9 models × 5 domains).